# Loading data

notebook on how to load in the relevant data sets in their full form:

'full form' means:

- SV Gauss coefficient time series
- The full (resolution-spanned) time record
- $N_{max} = 20$

'relevant datasets' include:

- CHAOS-8.6 
- Synthetic records, including:
    - pre-resolution (i.e. exact)
    - post-resolution

## Preamble



In [1]:
# setting up file path for imports
import sys
import os
from pathlib import Path

cwd = Path(os.getcwd())
print(cwd)
PROJECT_ROOT = cwd.resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

# installing relevant libraries
import chaosmagpy as cp
import numpy as np
import matplotlib.pyplot as plt
import h5py

# Project paths / utilities
from src.msc_thesis.paths import *
from src.msc_thesis.synSetup import *
from src.msc_thesis.synUtils import *
from src.msc_thesis.synDMD import *

/home/elliott/projects/msc_thesis_project/Last_stretch/Data_prep
['/home/elliott/projects/msc_thesis_project', '/home/elliott/miniforge3/envs/msc_thesis/lib/python311.zip', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/lib-dynload', '', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/site-packages', '/home/elliott/projects/msc_thesis_project']
['/home/elliott/projects/msc_thesis_project', '/home/elliott/projects/msc_thesis_project', '/home/elliott/miniforge3/envs/msc_thesis/lib/python311.zip', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/lib-dynload', '', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/site-packages', '/home/elliott/projects/msc_thesis_project']


/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/site-packages/chaosmagpy/model_utils.py:795: UserWarning: Input coordinates include the poles.
  warnings.warn('Input coordinates include the poles.')


['/home/elliott/projects/msc_thesis_project', '/home/elliott/projects/msc_thesis_project', '/home/elliott/projects/msc_thesis_project', '/home/elliott/miniforge3/envs/msc_thesis/lib/python311.zip', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/lib-dynload', '', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/site-packages', '/home/elliott/projects/msc_thesis_project']
C has shape dimensions: (148, 440, 440)


## Defining full record

The full record is defined as that which includes the maximal information available from CHAOS-8.6, that is compatible with the format used in the resolution output.


### Time parameters

Resolution outputs have the temporal parameters of:

- 29.4 year record range (1997.1-2026.5)
- 0.2 year gauss coefficient knot spacing
- 148 time step data points

These specifications are confirmed by examining the resolution file provided by DTU directly (below)

In [3]:
f = h5py.File((f'{CHAOS_RESOL_DIR}/CHAOS_Resol_1997_2026_0806_SV.h5'),"r")


# getting time parameters
# array of time steps (decimal year)
times_res = np.asarray(f['tp'])
# value of temporal sampling spacing
dt_res = np.asarray(f["dt"])
# number of time points in total
Nt_res = np.asarray(f["n_tp"])

# converting to mjd (to withdraw from ChaosMagPy)
times_res_mjd = cp.dyear_to_mjd(times_res)

print(f"The resolved output format spans {np.min(times_res)}-{np.max(times_res)} \n\
With a gauss coefficient knot spacing of {dt_res} years \n\
Resulting in {Nt_res} time sample points per time series")

f.close()

The resolved output format spans 1997.1-2026.5 
With a gauss coefficient knot spacing of 0.2 years 
Resulting in 148 time sample points per time series


## Withdrawing data over this record

### CHAOS-8.6 data

CHAOS-8.6 SV Gauss coefficients can be withdrawn directly using ChaosMagPy.

In [11]:
def CHAOS_Full_SV_Record_Obtain(model_version="CHAOS-8.6.mat"):
    # load the CHAOS model from the mat-file
    model = cp.load_CHAOS_matfile(f'{CHAOS_DIR}/{model_version}')

    # printing full unbounded extent of CHAOS8.6 model (decimal year)
    print('Full CHAOS-8.6 timespan is:', cp.mjd_to_dyear(model.model_tdep.breaks[[0, -1]]))

    # gauss coeffs in natural order, i.e. g(n,m): g(1,0), g(1, 1), h(1, 1), ...
    gnm_chaos = model.synth_coeffs_tdep(times_res_mjd, nmax=20, deriv=1)  # shape: (10, 224)

    return(gnm_chaos)

gnm_chaos = CHAOS_Full_SV_Record_Obtain()

print(f"The CHAOS-8.6 data set stores the time series data for:\n\
{np.shape(gnm_chaos)[1]} gauss coefficients\n\
at {np.shape(gnm_chaos)[0]} time steps")


Full CHAOS-8.6 timespan is: [1997.1 2026.6]
The CHAOS-8.6 data set stores the time series data for:
440 gauss coefficients
at 148 time steps


###  Synthetic data

A data record with identical temporal parameters can be constructed for the synthetic records.

Note start time here arbitrary - decision to make start of chaos record 0 was made.

In [ ]:
times_res_relative = times_res - times_res[0]

# define function - pulls out signals necessary + input mode info for comparison
def Synthetic_Full_SV_Record_Obtain(mode_numbers, times=times_res_relative):
    # for each mode - forming the ideal input record and withdrawing precomputed resolved record
    spline_path = Path(FELIX_DIR) / "R_splines_arbitrary.h5"
    synthetic_suite_info = {}
    gnm_total_ideal_list = []
    gnm_total_resolved_list = []

    with h5py.File(spline_path, "r") as h5_file:
        for mode_number in tqdm(mode_numbers, desc="Loading combined modes"):

            # getting amplitude scaler
            amplitude_scaler = mode_amp_scalings[mode_number]


            # 1) getting ideal mode gauss coefficient time series
            mode_data = Component_Load_SV(mode_number)
            eigenvalue = mode_data["eigenvalue"]
            ideal_gnm_phasor = amplitude_scaler * mode_data["gnm"]
            true_period = 2.0 * np.pi / np.abs(eigenvalue.imag)
            gnm_mode_ideal = G_Time_Series_Eval(
                ideal_gnm_phasor,
                eigenvalue,
                times
            )

            # 2) getting the resolved gauss coefficient time series
            dataset_name = f"mode_{mode_number}/without_decay"
            gnm_spline = np.asarray(h5_file[dataset_name][()])
            gnm_mode_resolved = H_sv @ (
                amplitude_scaler * gnm_spline
            )

            # store it all for future reference
            synthetic_suite_info[mode_number] = {
                "true_period": float(true_period),
                "true_eigenvalue": eigenvalue,
                "gnm_phasor": ideal_gnm_phasor,
                "gnm_ideal": gnm_mode_ideal,
                "gnm_resolved": gnm_mode_resolved,
            }
            gnm_total_ideal_list.append(gnm_mode_ideal)
            gnm_total_resolved_list.append(gnm_mode_resolved)

    # stacking lists to get cumulative wave signals
    gnm_total_ideal = np.sum(np.asarray(gnm_total_ideal_list), axis=0)
    gnm_total_resolved = np.sum(
        np.asarray(gnm_total_resolved_list),
        axis=0,
    )

    # deleting deprecated objects
    del gnm_total_ideal_list
    del gnm_total_resolved_list

    return(gnm_total_ideal, gnm_total_resolved, synthetic_suite_info)


In [ ]:
# selecting a set of input modes
mode_numbers = np.arange(1, 63, 10)
mode_numbers = [str(mode_number) for mode_number in mode_numbers]


gnm_syn_ideal, gnm_syn, syn_suite_info = Synthetic_Full_SV_Record_Obtain(mode_numbers, times_res_relative)

print(f"The ideal synthetic data set stores the time series data for:\n\
{np.shape(gnm_syn_ideal)[1]} gauss coefficients\n\
at {np.shape(gnm_syn_ideal)[0]} time steps")

print(f"The resolved synthetic data set stores the time series data for:\n\
{np.shape(gnm_syn)[1]} gauss coefficients\n\
at {np.shape(gnm_syn)[0]} time steps")


Loading combined modes: 100%|██████████| 7/7 [00:00<00:00, 42.13it/s]

The ideal synthetic data set stores the time series data for:
440 gauss coefficients
at 148 time steps
The resolved synthetic data set stores the time series data for:
440 gauss coefficients
at 148 time steps
